# Input ports, boundary lifts and passivity

Read the [derivation and limitations](INPUT_PORTS_AND_PASSIVITY.md). This independently authored illustration responds to the B/H/E input and passivity questions raised by the 2024 unit-cell paper ([DOI](https://doi.org/10.1109/TMAG.2024.3486683)). The original paper and supplied notebooks are not redistributed or executed here.

We distinguish matched terminal problems, coefficient correspondence, and passivity. No FE solve, shifted-synthesis certification, performance claim or new numerical acceptance threshold is added.

## Exact boundary lift and matched index shift

The static lift has the nonzero boundary trace. Later corrections have zero trace. For the same normalized conjugate port, $Y_B=S/s$ and $Z_H=s/S$. The unshifted recurrences agree exactly after shifting indices by two:

$$L_{-1}^{B}=L_1^{H},\quad R_0^{B}=R_2^{H},\quad L_1^{B}=L_3^{H},\quad\ldots$$

The checker compares constrained B and full-space H recurrences against scalar rational extraction from the independently inverted transfer. The claim that coefficients can differ applies to a **different/mismatched port**, where the toy gives $L_1=3/5$, rather than the matched $5/17$. The response $Y_B-c/s$ is the matched E-type response, not an arbitrary electric excitation. Operator-space details for the paper's compact equation (13) are in the [note](INPUT_PORTS_AND_PASSIVITY.md).

In [3]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'validation').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / 'validation'))
from input_ports_exact import run
# Reads the source-bound Mathematica record and runs an independent Fraction Bessel oracle.
data = run()
for name in ['static_lift', 'static_energy_c', 'L_excitation', 'different_port_L1', 'Y_B', 'Y_E_response', 'Z_H_same_port']:
    print(f"{name}: {data['port'][name]}")
matched = data['port']['matched_index_shift']
for b, h, vb, vh, scalar in zip(matched['B_labels'], matched['H_labels'], matched['B_elements'], matched['H_elements'], matched['independent_scalar_elements']):
    print(f"B {b} = H {h}: {vb[1]} = {vh[1]}; scalar extraction {scalar[1]}")
print('Literal full compact product minus correct response:', data['port']['literal_full_minus_response'])


static_lift: ['1', '-1/5', '-2/5']
static_energy_c: 17/5
L_excitation: 5/17
different_port_L1: 3/5
Y_B: (6*s**3 + 37*s**2 + 52*s + 17)/(6*s**3 + 13*s**2 + 5*s)
Y_E_response: (30*s**2 + 83*s + 39)/(30*s**2 + 65*s + 25)
Z_H_same_port: (6*s**3 + 13*s**2 + 5*s)/(6*s**3 + 37*s**2 + 52*s + 17)
B L_-1 = H L_1: 5/17 = 5/17; scalar extraction 5/17
B R_0 = H R_2: 25/39 = 25/39; scalar extraction 25/39
B L_1 = H L_3: 460/1521 = 460/1521; scalar extraction 460/1521
B R_2 = H R_4: 1058/4251 = 1058/4251; scalar extraction 1058/4251
B L_3 = H L_5: 828/11881 = 828/11881; scalar extraction 828/11881
B R_4 = H R_6: 12/109 = 12/109; scalar extraction 12/109
Literal full compact product minus correct response: (-900*s**5 - 4980*s**4 - 9229*s**3 - 6474*s**2 - 1521*s)/(900*s**5 + 7500*s**4 + 20575*s**3 + 24075*s**2 + 12025*s + 2125)


## Independent circular-wire check

Maxwell integrations generate five electric and four magnetic modes. A/H and T/E mappings retain history terms and boundary traces. A separate Bessel power series checks the nine-element unshifted terminal fraction through $z^8$. These are per-unit-length quantities with $a=\mu=\sigma=1$, $R_{dc}=1/\pi$ and $L_{dc}=1/(8\pi)$. They do not certify arbitrary ports or 3D FE convergence.

A/H and T/E differentiation maps are construction-consistency identities, not independent FE solves. The Bessel-series comparison is independent of the recurrence. The physical Faraday equation is $E_z'=sB_\theta$; normalized recurrence corrections use their stage multiplier instead.

In [5]:
for name in ['E_even', 'H_odd', 'R_even', 'L_odd', 'Z_over_Rdc', 'Bessel_series_through_z8']:
    print(f"{name}: {data['wire'][name]}")

E_even: ['1', '2*r**2 - 1', '6*r**4 - 6*r**2 + 1', '20*r**6 - 30*r**4 + 12*r**2 - 1', '70*r**8 - 140*r**6 + 90*r**4 - 20*r**2 + 1']
H_odd: ['r/(2*pi)', '3*r**3/(2*pi) - r/pi', '5*r**5/pi - 6*r**3/pi + 3*r/(2*pi)', '35*r**7/(2*pi) - 30*r**5/pi + 15*r**3/pi - 2*r/pi']
R_even: ['1/pi', '3/pi', '5/pi', '7/pi', '9/pi']
L_odd: ['1/(8*pi)', '1/(16*pi)', '1/(24*pi)', '1/(32*pi)']
Z_over_Rdc: (25*z**4 + 9600*z**3 + 846720*z**2 + 20643840*z + 92897280)/(z**4 + 1200*z**3 + 201600*z**2 + 9031680*z + 92897280)
Bessel_series_through_z8: -1133*z**8/2446118092800 + 647*z**7/95126814720 - 11*z**6/110100480 + 13*z**5/8847360 - z**4/46080 + z**3/3072 - z**2/192 + z/8 + 1


## Counterexamples and numerical safeguards

An independently specified magnetic surface value can violate Ampere's law for a fixed current distribution; regularity removes the $C/r$ freedom. Euclidean remainders need not be positive. A positive shifted branch $qL$ need not be passive in physical $s$. Floating-point breakdown and orthogonality need separate diagnostics; positive reported coefficients alone are not an accuracy certificate.

The Type1 $Z/s$ and Type2 $Z$ conventions remain those of the [existing expansion study](01_expansion_and_energy.ipynb). This notebook does not establish that power divided by $s$ universally restores passivity.

In [7]:
for name, value in data['caveats'].items():
    print(f"{name}: {value}")
print('Exact checks completed; no FE, general shifted theorem, or independent reviewer verdict claimed.')

regular_Ampere_H: r/2
incompatible_adjusted_H: r/2 - 1/2
Ampere_defect: -1/(2*r)
shifted_branch: s - 1
shifted_branch_physical_DC: -1
Euclidean_remainder: -2
Exact checks completed; no FE, general shifted theorem, or independent reviewer verdict claimed.


## Source hash convention

`source_sha256_lf` uses UTF-8 without BOM and LF-normalized source, preserving the final newline. It is distinct from raw CRLF worktree bytes. Submission manifests label Git-blob SHA256 and raw-worktree SHA256 separately. The checker uses explicit exceptions, including under `python -O`; CI declares its SymPy version. This notebook checks the saved record without overwriting it.